# 4 · Functional Equivalence — does the fine-tuned model perform the same act?

**Measuring Pragmatic Alignment in LLM-Based Agents**
University of Trier · NLP Master's Program · WS 2025/26

This notebook answers the project's central question directly.

Notebook 1 showed that the fine-tuned Qwen model matches human *style* — sentiment
and grammatical profile are near-identical — while sharing almost no vocabulary
(Jaccard 0.022) and only moderate meaning (SBERT 0.456) with the human reply to
the same tweet. What it could not show is whether the model performs the same
**communicative act**: where a human opposed, does the model oppose? Where a human
issued a command, does the model command?

The annotated corpus gives us gold pragmatic labels for the *authentic* reply of
each item. It does not give us labels for the model's reply. We therefore label
the model's replies automatically with the strongest classifier from notebook 2,
and compare those predictions against the human gold labels for the same item.

| Step | What it does |
|------|-------------|
| 1 | Join each annotated item to the fine-tuned model's reply from the source corpus |
| 2 | Train the notebook 2 classifier (RoBERTa-base, reply-only) and verify it reproduces |
| 3 | Predict the four pragmatic labels for every fine-tuned reply |
| 4 | Match rate per dimension, against a chance baseline, with Cohen's kappa |
| 5 | Joint agreement — all four dimensions at once, and three of four |
| 6 | Confusion matrices, to see the *direction* of disagreement |

Everything is reported twice: on all annotated items, and on the 120 held-out
items alone. **The held-out figure is the defensible one** — the classifier was
trained on 560 of the annotated items, so its predictions there are optimistic.

In [1]:
import ast
import csv
import copy
import json
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
csv.field_size_limit(10**9)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("..")
LABEL_COLS = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"seed={SEED}  device={DEVICE}")

seed=42  device=mps


---
## 1. Join the fine-tuned replies to the annotated items

The annotated file holds the target tweet, the authentic reply and its gold
labels. The fine-tuned model's reply to the same tweet lives in the source
corpus, so the two are matched on the (tweet, reply) pair using the same
normalisation the preparation script uses.

In [2]:
sys.path.insert(0, str(ROOT / "src"))
from prepare_data import match_key  # noqa: E402

ann = pd.read_csv(ROOT / "data" / "annotated_clean.csv")

def last_user_turn(prompt):
    try:
        for turn in reversed(ast.literal_eval(prompt)):
            if turn.get("role") == "user":
                return turn.get("content")
    except (ValueError, SyntaxError):
        pass
    return None

by_pair, by_reply = {}, {}
with (ROOT / "data" / "dataset.english.csv").open(encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        tweet = last_user_turn(row["prompt"])
        if not tweet:
            continue
        k_tweet, k_reply = match_key(tweet), match_key(row["authentic_reply"])
        by_pair[(k_tweet, k_reply)] = row["ft_model_reply"]
        by_reply.setdefault(k_reply, []).append(row["ft_model_reply"])

ft_replies = []
for _, item in ann.iterrows():
    k_tweet, k_reply = match_key(item["target_tweet"]), match_key(item["authentic_reply"])
    hit = by_pair.get((k_tweet, k_reply))
    if hit is None:
        candidates = by_reply.get(k_reply, [])
        hit = candidates[0] if len(candidates) == 1 else None
    ft_replies.append(hit)

ann["ft_model_reply"] = ft_replies
matched = ann["ft_model_reply"].notna()
print(f"annotated items          : {len(ann)}")
print(f"matched to a model reply : {matched.sum()}")
print(f"unmatched (dropped)      : {(~matched).sum()}")
ann = ann[matched].reset_index(drop=True)

print(f"\nmean length — authentic {ann.authentic_reply.str.len().mean():.1f} chars, "
      f"fine-tuned model {ann.ft_model_reply.str.len().mean():.1f} chars")

annotated items          : 800
matched to a model reply : 799
unmatched (dropped)      : 1

mean length — authentic 99.6 chars, fine-tuned model 80.3 chars


---
## 2. The classifier

The same architecture, recipe and split as notebook 2: RoBERTa-base with four
classification heads, reply-only input, ten epochs, weights restored from the
epoch with the best validation macro-F1. Its test-set scores are printed so they
can be checked against notebook 2 (0.767 accuracy / 0.627 macro-F1).

In [3]:
encoders, Y = {}, pd.DataFrame(index=ann.index)
for col in LABEL_COLS:
    le = LabelEncoder().fit(ann[col])
    encoders[col] = le
    Y[col] = le.transform(ann[col])

joint = ann[LABEL_COLS].agg("|".join, axis=1)
strata = joint.where(joint.map(joint.value_counts()) >= 8, "RARE")
idx_train, idx_temp = train_test_split(ann.index, test_size=0.30, random_state=SEED, stratify=strata)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=strata[idx_temp])
print(f"train {len(idx_train)} · val {len(idx_val)} · test {len(idx_test)}")

train 559 · val 120 · test 120


In [4]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "roberta-base"
MAX_LEN, BATCH, EPOCHS, LR = 128, 16, 10, 2e-5
NUM_CLASSES = [len(encoders[c].classes_) for c in LABEL_COLS]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReplyDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else np.asarray(labels)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, padding="max_length",
                        max_length=MAX_LEN, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

class MultiHeadTransformer(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = torch.nn.Dropout(0.1)
        self.heads = torch.nn.ModuleList(
            [torch.nn.Linear(self.encoder.config.hidden_size, n) for n in NUM_CLASSES])
    def forward(self, input_ids, attention_mask, **kw):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return [head(self.dropout(pooled)) for head in self.heads]

torch.manual_seed(SEED)
model = MultiHeadTransformer().to(DEVICE)
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = torch.nn.CrossEntropyLoss()

train_dl = DataLoader(ReplyDataset(ann.loc[idx_train, "authentic_reply"], Y.loc[idx_train].values),
                      batch_size=BATCH, shuffle=True)
val_dl = DataLoader(ReplyDataset(ann.loc[idx_val, "authentic_reply"], Y.loc[idx_val].values), batch_size=32)
test_dl = DataLoader(ReplyDataset(ann.loc[idx_test, "authentic_reply"], Y.loc[idx_test].values), batch_size=32)

def predict(loader):
    model.eval()
    out = [[] for _ in LABEL_COLS]
    with torch.no_grad():
        for batch in loader:
            batch.pop("labels", None)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            for i, logit in enumerate(model(**batch)):
                out[i].append(logit.argmax(-1).cpu().numpy())
    return {c: np.concatenate(out[i]) for i, c in enumerate(LABEL_COLS)}

best_f1, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for batch in train_dl:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optim.zero_grad()
        logits = model(**batch)
        loss = sum(loss_fn(logits[i], labels[:, i]) for i in range(len(LABEL_COLS)))
        loss.backward()
        optim.step()
        total += loss.item()
    vp = predict(val_dl)
    val_f1 = float(np.mean([f1_score(Y.loc[idx_val, c], vp[c], average="macro", zero_division=0)
                            for c in LABEL_COLS]))
    mark = ""
    if val_f1 > best_f1:
        best_f1, best_state, mark = val_f1, copy.deepcopy(model.state_dict()), "   <- best"
    print(f"  epoch {epoch+1:2}/{EPOCHS}  loss {total/len(train_dl):.4f}  val macro-F1 {val_f1:.4f}{mark}")

model.load_state_dict(best_state)
tp = predict(test_dl)
acc = np.mean([accuracy_score(Y.loc[idx_test, c], tp[c]) for c in LABEL_COLS])
mf1 = np.mean([f1_score(Y.loc[idx_test, c], tp[c], average="macro", zero_division=0) for c in LABEL_COLS])
print(f"\nclassifier held-out performance: accuracy {acc:.4f}  macro-F1 {mf1:.4f}")
print("(notebook 2 reports 0.767 / 0.627; RoBERTa fine-tuning on MPS varies by ~0.03 between runs)")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch  1/10  loss 3.3841  val macro-F1 0.2910   <- best


  epoch  2/10  loss 2.8772  val macro-F1 0.3656   <- best


  epoch  3/10  loss 2.3370  val macro-F1 0.4926   <- best


...

  epoch  7/10  loss 0.7486  val macro-F1 0.5530


  epoch  8/10  loss 0.4970  val macro-F1 0.6002


  epoch  9/10  loss 0.3379  val macro-F1 0.5966


  epoch 10/10  loss 0.2473  val macro-F1 0.6158   <- best



classifier held-out performance: accuracy 0.7625  macro-F1 0.5902
(notebook 2 reports 0.767 / 0.627; RoBERTa fine-tuning on MPS varies by ~0.03 between runs)


---
## 3. Label the fine-tuned model's replies

The classifier now reads each model-generated reply and assigns it the four
pragmatic labels, exactly as it would for a human reply.

In [5]:
ft_dl = DataLoader(ReplyDataset(ann["ft_model_reply"].astype(str)), batch_size=32)
ft_pred = predict(ft_dl)
for col in LABEL_COLS:
    ann[f"ft_pred_{col}"] = encoders[col].inverse_transform(ft_pred[col])

print("Predicted label distribution for the fine-tuned replies, "
      "against the authentic gold distribution:\n")
for col in LABEL_COLS:
    gold_share = ann[col].value_counts(normalize=True)
    pred_share = ann[f"ft_pred_{col}"].value_counts(normalize=True)
    comparison = pd.DataFrame({"authentic gold": gold_share, "model predicted": pred_share}).fillna(0)
    print(f"{col}")
    print((comparison * 100).round(1).to_string(), "\n")

Predicted label distribution for the fine-tuned replies, against the authentic gold distribution:

STANCE
         authentic gold  model predicted
NEUTRAL            30.7             22.2
OPPOSE             45.1             55.2
SUPPORT            24.3             22.7 

ACTION
           authentic gold  model predicted
STATEMENT            71.5             80.6
QUESTION             14.9             12.9
COMMAND              11.0              3.8
REACTION              2.6              2.8 

PERSONALNESS
          authentic gold  model predicted
GENERAL             84.2             84.9
PERSONAL            15.8             15.1 

POLITENESS
        authentic gold  model predicted
NORMAL            77.3             76.6
RUDE              19.5             23.0
POLITE             3.1              0.4 



---
## 4. Per-dimension agreement

For each dimension: how often the model's reply carries the same label as the
human's, what that rate would be by chance if the two label sets were
independent draws from their own marginal distributions, and Cohen's kappa,
which is the raw rate corrected for that chance level.

In [6]:
def chance_rate(gold, pred):
    """Expected agreement if the two were independent draws from their marginals."""
    classes = sorted(set(gold) | set(pred))
    g = pd.Series(gold).value_counts(normalize=True)
    p = pd.Series(pred).value_counts(normalize=True)
    return float(sum(g.get(c, 0.0) * p.get(c, 0.0) for c in classes))

def agreement_table(subset, name):
    rows = []
    for col in LABEL_COLS:
        gold, pred = subset[col].values, subset[f"ft_pred_{col}"].values
        observed = float((gold == pred).mean())
        expected = chance_rate(gold, pred)
        rows.append({
            "Dimension": col,
            "Match rate": observed,
            "Chance rate": expected,
            "Above chance": observed - expected,
            "Cohen's kappa": cohen_kappa_score(gold, pred),
        })
    table = pd.DataFrame(rows).set_index("Dimension")
    table.loc["MEAN"] = table.mean()
    print(f"\n=== {name} (n={len(subset)}) ===")
    print(table.round(3).to_string())
    return table

full_tbl = agreement_table(ann, "All annotated items — classifier saw 70% of these in training")
test_sub = ann.loc[idx_test]
test_tbl = agreement_table(test_sub, "Held-out items only — the defensible figure")


=== All annotated items — classifier saw 70% of these in training (n=799) ===
              Match rate  Chance rate  Above chance  Cohen's kappa
Dimension                                                         
STANCE             0.527        0.372         0.155          0.247
ACTION             0.627        0.600         0.027          0.067
PERSONALNESS       0.788        0.739         0.050          0.191
POLITENESS         0.677        0.638         0.040          0.109
MEAN               0.655        0.587         0.068          0.154

=== Held-out items only — the defensible figure (n=120) ===
              Match rate  Chance rate  Above chance  Cohen's kappa
Dimension                                                         
STANCE             0.542        0.373         0.169          0.269
ACTION             0.650        0.572         0.078          0.182
PERSONALNESS       0.817        0.769         0.048          0.207
POLITENESS         0.683        0.631         0.052     

---
## 5. Joint agreement

A reply is functionally equivalent only if it performs the same act on *every*
dimension, so the per-dimension rates above are an optimistic view. The chance
baselines here assume the four dimensions are independent.

In [7]:
def poisson_binomial_at_least(probs, k):
    """P(at least k successes) for independent Bernoulli trials."""
    dist = np.zeros(len(probs) + 1)
    dist[0] = 1.0
    for p in probs:
        dist[1:] = dist[1:] * (1 - p) + dist[:-1] * p
        dist[0] *= (1 - p)
    return float(dist[k:].sum())

def joint_table(subset, name):
    hits = np.zeros(len(subset), dtype=int)
    for col in LABEL_COLS:
        hits += (subset[col].values == subset[f"ft_pred_{col}"].values).astype(int)
    chances = [chance_rate(subset[c].values, subset[f"ft_pred_{c}"].values) for c in LABEL_COLS]

    rows = []
    for k, label in [(4, "All 4 dimensions match"), (3, "At least 3 of 4 match")]:
        observed = float((hits >= k).mean())
        expected = poisson_binomial_at_least(chances, k)
        rows.append({"Criterion": label, "Observed": observed, "Chance": expected,
                     "Ratio to chance": observed / expected if expected else np.nan})
    table = pd.DataFrame(rows).set_index("Criterion")
    print(f"\n=== {name} (n={len(subset)}) ===")
    print(table.round(3).to_string())
    print("\n  number of dimensions matched, distribution:")
    for k in range(5):
        n = int((hits == k).sum())
        print(f"    {k}/4 : {n:4}  ({n/len(subset):5.1%})")
    return table, hits

full_joint, full_hits = joint_table(ann, "All annotated items")
test_joint, test_hits = joint_table(test_sub, "Held-out items only")


=== All annotated items (n=799) ===
                        Observed  Chance  Ratio to chance
Criterion                                                
All 4 dimensions match     0.169   0.105            1.609
At least 3 of 4 match      0.572   0.449            1.273

  number of dimensions matched, distribution:
    0/4 :    7  ( 0.9%)
    1/4 :   83  (10.4%)
    2/4 :  252  (31.5%)
    3/4 :  322  (40.3%)
    4/4 :  135  (16.9%)

=== Held-out items only (n=120) ===
                        Observed  Chance  Ratio to chance
Criterion                                                
All 4 dimensions match     0.158   0.103            1.532
At least 3 of 4 match      0.650   0.446            1.456

  number of dimensions matched, distribution:
    0/4 :    2  ( 1.7%)
    1/4 :   10  ( 8.3%)
    2/4 :   30  (25.0%)
    3/4 :   59  (49.2%)
    4/4 :   19  (15.8%)


---
## 6. Direction of disagreement

The match rate says how often the model diverges; the confusion matrices say
*how*. Rows are the label a human's reply carried, columns the label predicted
for the model's reply to the same tweet.

In [8]:
for col in LABEL_COLS:
    cm = pd.crosstab(ann[col], ann[f"ft_pred_{col}"],
                     rownames=[f"authentic {col}"], colnames=["model predicted"], dropna=False)
    for cls in encoders[col].classes_:
        if cls not in cm.columns:
            cm[cls] = 0
    cm = cm[list(encoders[col].classes_)]
    print(f"\n=== {col} (all {len(ann)} items) ===")
    print(cm.to_string())
    share = (cm.div(cm.sum(axis=1), axis=0) * 100).round(1)
    print("\nrow-normalised (% of each authentic class):")
    print(share.to_string())


=== STANCE (all 799 items) ===
model predicted   NEUTRAL  OPPOSE  SUPPORT
authentic STANCE                          
NEUTRAL                69     117       59
OPPOSE                 56     267       37
SUPPORT                52      57       85

row-normalised (% of each authentic class):
model predicted   NEUTRAL  OPPOSE  SUPPORT
authentic STANCE                          
NEUTRAL              28.2    47.8     24.1
OPPOSE               15.6    74.2     10.3
SUPPORT              26.8    29.4     43.8

=== ACTION (all 799 items) ===
model predicted   COMMAND  QUESTION  REACTION  STATEMENT
authentic ACTION                                        
COMMAND                10        11         1         66
QUESTION                3        21         2         93
REACTION                0         2         2         17
STATEMENT              17        69        17        468

row-normalised (% of each authentic class):
model predicted   COMMAND  QUESTION  REACTION  STATEMENT
authentic ACTION 

---
## 7. Caveats

Two limitations bound what this measurement can support, and both belong in any
write-up of it.

**(a) The model's labels are predicted, not human-annotated.** Every figure above
compares a *human gold* label against a *classifier's guess*. The classifier's
own mean macro-F1 is roughly 0.63 on held-out human replies, so a share of every
disagreement reported here is classifier error rather than genuine pragmatic
divergence. The measure therefore bounds functional equivalence from below and
its resolution is limited by the critic. Hand-annotating a sample of the
fine-tuned replies would be the way to separate the two sources of error.

**(b) The classifier was trained on human replies and is applied to machine
replies.** That is a distribution shift, and we cannot quantify it: the
hand-annotated labels for the fine-tuned replies produced for Deliverable 2 are
no longer available, so there is no held-out set on which to measure how well the
classifier transfers. Notebook 1 gives indirect reassurance — a detector
distinguishing human from fine-tuned replies performs at chance on word choice
(0.496), so the two are lexically similar — but that is not a validation of
classifier transfer, and the possibility that the classifier is systematically
worse on machine text cannot be excluded.

A third, smaller point: the per-dimension chance baselines are computed from the
marginals of each subset, and the joint baselines additionally assume the four
dimensions are independent. They are not entirely independent in the gold data,
so the joint chance figures are approximate.

In [9]:
out_cols = (["target_tweet", "authentic_reply", "ft_model_reply"]
            + LABEL_COLS + [f"ft_pred_{c}" for c in LABEL_COLS])
export = ann[out_cols].copy()
for col in LABEL_COLS:
    export[f"match_{col}"] = (ann[col] == ann[f"ft_pred_{col}"])
export["dimensions_matched"] = full_hits
export["held_out"] = export.index.isin(idx_test)
export.to_csv("functional_equivalence.csv", index=False)
print(f"per-item results -> functional_equivalence.csv ({len(export)} rows)")

summary = pd.concat(
    [full_tbl.assign(subset="all items"), test_tbl.assign(subset="held-out only")]
).reset_index()
summary.to_csv("functional_equivalence_summary.csv", index=False)
print(f"summary tables   -> functional_equivalence_summary.csv")

per-item results -> functional_equivalence.csv (799 rows)
summary tables   -> functional_equivalence_summary.csv
